# Evaluation — Mycorrhizal Barcelona  →  Platanus Pollen-Allergen Priority
### CRISP-DM Phase 5 (Evaluation) · Group 4 · MaAI01 25-26

**Purpose.** Judge what we built in Sessions 1–4 against the success criteria from the
Session-1 brief, and produce a defensible verdict — *deploy / iterate / stop* — with the
evidence behind it. This notebook is the evidence; every number it prints appears,
identical, in `docs/evaluation-report.md`.

**Track — both, in sequence (this is the project's spine).**
We are a *Track A* team (we trained a model) **whose evaluation killed the model**, after
which we pivoted to a *Track B* analytical product (a conclusion/recommendation, no trained
predictor). So the notebook runs **Cycle A → the kill → Cycle B**:

| | Cycle A — the model | Cycle B — the shipped product |
|---|---|---|
| Artifact | linear regressor of a barrier-severity composite (R² 0.877) | two-layer composite indicator (pollen-source × residential exposure) |
| Core question | does it generalize *and serve the ecological decision*? | are the conclusions valid and act-on-able? |
| Verdict | **STOP** (falsified on independent data) | **SHIP ~75%**, deploy-pending stakeholder sign-off |

**Inputs (all frozen, read-only):** `docs/problem-brief.md` (Cycle-A criteria),
`phase-6/phase-1-audit.md §B` (Cycle-B numeric criteria),
`outputs/phase-4/predictions.parquet` + `model_artifact.joblib` (the model),
`data/processed/scored_grid.parquet` + `data/gbif-fungi-all.json` (the external test),
`data/processed/allergen_layers.parquet` (the pivot layers).

**Outputs:** `docs/evaluation-report.md` (verdict), `docs/failure-gallery.md` (Track A),
`docs/validity-audit.md` + `docs/conclusions-brief.md` (Track B), `docs/evaluation-log.md`.

In [1]:
# Cell 2 — imports, determinism, frozen artifacts
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import spearmanr

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Resolve repo ROOT whether the kernel starts in notebooks/ or the repo root.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), f"cannot locate repo root from {Path.cwd()}"
sys.path.insert(0, str(ROOT / "src"))

# Pure helpers from the canonical pipeline (importing does NOT run anything / write files).
import external_validation as ev       # OLS, partial-F, VIF for the external kill
import allergen_priority as ap         # minmax, topk, jaccard, burden_capture for T1-T4

preds   = pd.read_parquet(ROOT / "outputs/phase-4/predictions.parquet")
layers  = gpd.read_parquet(ROOT / "data/processed/allergen_layers.parquet")
grid    = gpd.read_parquet(ROOT / "data/processed/scored_grid.parquet")
stability = json.loads((ROOT / "outputs/phase-4/stability.json").read_text())

print(f"predictions: {preds.shape[0]} cells ({(preds.split=='test').sum()} held-out test)")
print(f"allergen layers: {layers.shape[0]} cells, {layers.shape[1]} cols")
print(f"scored grid: {grid.shape[0]} cells   |   seed = {RANDOM_SEED}")

predictions: 494 cells (88 held-out test)
allergen layers: 494 cells, 19 cols
scored grid: 494 cells   |   seed = 42


## The bar we set in Session 1

Evaluation grades against the brief, not against our hopes. We have **two** briefs because
the project pivoted:

**Cycle A — the original ecological claim** (`docs/problem-brief.md`). The brief's headline
sub-question (§6) was *"which zones rank highest on the multi-barrier composite"* with the
project's underlying motivation being **belowground ecological (mycorrhizal) recovery**. The
operative, falsifiable claim we built on: *the host-mycorrhizal (AM→EM) layer carries signal
about real fungal outcomes.* Criteria 1–7 are reproducibility/coverage/usability gates; the
**load-bearing scientific criterion is that the ecological signal is real.**

**Cycle B — the pivot** (`phase-6/phase-1-audit.md §B`), six pre-registered numeric criteria,
restated as thresholds before the build:

| # | Criterion | Threshold | Test |
|---|---|---|---|
| 1 | Exposure re-orders vs naive plane-density | top-15 Jaccard < 0.70 **and** Spearman < 0.90 | T1 |
| 2 | Both layers material; inputs not collinear | \|corr(pri,·)\| ≥ 0.3 each; \|corr(src,expo)\| < 0.8 | T2 |
| 3 | Beat density-only on burden captured | margin > 0 at top-15 | T3 |
| 4 | Verdict survives perturbation | T1 holds under 3 perturbations | T4 |
| 5 | Equity precondition (deprivation decorrelated) | \|r\| < 0.7 from both layers | V3-1 |
| 6 | Failure-and-pivot documented | exists on disk | file check |

In [2]:
# Cell 4 — results-vs-criteria table (Cycle B, the shipped product)
# Filled from the reproductions computed below (Cells B2-B5). Verdict in {met, partial, unmet}.
criteria = pd.DataFrame([
    {"#":1,"criterion":"exposure re-orders vs density","target":"J15<0.70 & rho<0.90","result":"0.30 / 0.89","verdict":"met"},
    {"#":2,"criterion":"both layers material, not collinear","target":">=0.3 / >=0.3 / <0.8","result":"0.80 / 0.64 / 0.30","verdict":"met"},
    {"#":3,"criterion":"beat density-only on burden","target":"margin>0 @top15","result":"+0.046","verdict":"met"},
    {"#":4,"criterion":"verdict survives perturbation","target":"3/3 hold","result":"3/3","verdict":"met"},
    {"#":5,"criterion":"equity precondition (decorrelation)","target":"|r|<0.7 both","result":"-0.008 / 0.17","verdict":"met"},
    {"#":6,"criterion":"failure-and-pivot documented","target":"on disk","result":"present","verdict":"met"},
    {"#":7,"criterion":"SOURCE validated vs measured pollen","target":"any open series","result":"NO open data","verdict":"unmet (un-evaluable)"},
])
print(criteria.to_markdown(index=False))
print("\nCycle B: 6 of 6 build criteria MET; the 7th is un-evaluable by absence of data (declared).")

|   # | criterion                           | target               | result             | verdict              |
|----:|:------------------------------------|:---------------------|:-------------------|:---------------------|
|   1 | exposure re-orders vs density       | J15<0.70 & rho<0.90  | 0.30 / 0.89        | met                  |
|   2 | both layers material, not collinear | >=0.3 / >=0.3 / <0.8 | 0.80 / 0.64 / 0.30 | met                  |
|   3 | beat density-only on burden         | margin>0 @top15      | +0.046             | met                  |
|   4 | verdict survives perturbation       | 3/3 hold             | 3/3                | met                  |
|   5 | equity precondition (decorrelation) | |r|<0.7 both         | -0.008 / 0.17      | met                  |
|   6 | failure-and-pivot documented        | on disk              | present            | met                  |
|   7 | SOURCE validated vs measured pollen | any open series      | NO open data       | unmet 

---
# CYCLE A (Track A) — the model we built, evaluated, and stopped

In Session 4 we trained an interpretable linear regressor of the barrier-severity composite
`composite_score_B` and reported **test R² = 0.877**, judged the most methodologically mature
Phase-4 in the cohort. Evaluation starts where that headline ends — by asking *where the
average hides a failure*, and then by testing the claim against **data the model never saw.**

In [3]:
# Cell A1 — reproduce the Session-4 headline on the sacred test cluster (n=88)
def metrics(y, yhat):
    y, yhat = np.asarray(y,float), np.asarray(yhat,float)
    rss = float(((y-yhat)**2).sum()); tss = float(((y-y.mean())**2).sum())
    return dict(R2=1-rss/tss, MAE=float(np.abs(y-yhat).mean()),
                RMSE=float(np.sqrt(((y-yhat)**2).mean())))

te = preds[preds.split=="test"]
rows=[]
for est in ["LinearRegression","BaselineSpatialNearest","BaselineMean","BaselineDomainHeuristic"]:
    m = metrics(te["y_true"], te[f"y_pred__{est}"]); m["estimator"]=est; rows.append(m)
tbl = pd.DataFrame(rows)[["estimator","R2","MAE","RMSE"]].round(4)
print(tbl.to_markdown(index=False))
lr = tbl.iloc[0]
print(f"\nPre-registered pass (beat all baselines on test R2 AND MAE): "
      f"LR R2 {lr.R2} vs best baseline {tbl.R2[1:].max():.3f}; "
      f"LR MAE {lr.MAE} vs best baseline {tbl.MAE[1:].min():.3f}  ->  PASS")

| estimator               |      R2 |    MAE |   RMSE |
|:------------------------|--------:|-------:|-------:|
| LinearRegression        |  0.8769 | 0.0106 | 0.0509 |
| BaselineSpatialNearest  | -0.2898 | 0.1298 | 0.1649 |
| BaselineMean            | -0.616  | 0.1424 | 0.1845 |
| BaselineDomainHeuristic | -0.6217 | 0.1428 | 0.1849 |

Pre-registered pass (beat all baselines on test R2 AND MAE): LR R2 0.8769 vs best baseline -0.290; LR MAE 0.0106 vs best baseline 0.130  ->  PASS


In [4]:
# Cell A2 — where it fails: per-district residuals on the held-out cluster
te2 = te.copy()
te2["resid"] = te2["y_true"] - te2["y_pred__LinearRegression"]
perdist = (te2.groupby("district")["resid"]
           .agg(n="size", mean_resid="mean",
                abs_mean=lambda s: s.abs().mean(),
                max_abs=lambda s: s.abs().max()).round(4))
print(perdist.to_markdown())
print(f"\nWorst single cell |residual| = {te2['resid'].abs().max():.4f} "
      f"(district = {te2.loc[te2['resid'].abs().idxmax(),'district']}).")
print("District-level mean|resid| < 0.10 everywhere (OOD gate passes) — the failure is one "
      "spectacular cell, not a biased district.")

| district              |   n |   mean_resid |   abs_mean |   max_abs |
|:----------------------|----:|-------------:|-----------:|----------:|
| LES CORTS             |  37 |       0.0003 |     0.0011 |    0.0098 |
| SARRIÀ - SANT GERVASI |  51 |       0.0107 |     0.0174 |    0.3339 |

Worst single cell |residual| = 0.3339 (district = SARRIÀ - SANT GERVASI).
District-level mean|resid| < 0.10 everywhere (OOD gate passes) — the failure is one spectacular cell, not a biased district.


In [5]:
# Cell A3 — stress tests: drop each feature, does it crash or degrade?
import joblib
art = joblib.load(ROOT / "outputs/phase-4/model_artifact.joblib")
pipe, FEATURES, TARGET = art["model"], art["features"], art["target"]
test_full = gpd.read_parquet(ROOT / "data/splits/test.parquet")
Xte, yte = test_full[FEATURES].copy(), test_full[TARGET].to_numpy(float)
base_mae = float(np.abs(yte - pipe.predict(Xte)).mean())
print(f"intact test MAE = {base_mae:.4f}\n")
print("drop-feature stress (median imputer fills the gap — graceful-degradation test):")
rows=[]
for f in FEATURES:
    Xs = Xte.copy(); Xs[f] = np.nan
    try:
        mae = float(np.abs(yte - pipe.predict(Xs)).mean())
        rows.append((f, round(mae,4), "OK" if mae < 0.03 else "DEGRADED", round(mae-base_mae,4)))
    except Exception as e:
        rows.append((f, float("nan"), f"CRASH:{type(e).__name__}", float("nan")))
print(pd.DataFrame(rows, columns=["dropped_feature","MAE","flag","delta_vs_intact"]).to_markdown(index=False))

intact test MAE = 0.0106

drop-feature stress (median imputer fills the gap — graceful-degradation test):
| dropped_feature   |    MAE | flag     |   delta_vs_intact |
|:------------------|-------:|:---------|------------------:|
| mean_sealed       | 0.1205 | DEGRADED |            0.11   |
| mean_ndvi         | 0.0421 | DEGRADED |            0.0315 |
| lst_anomaly       | 0.0297 | OK       |            0.0192 |
| am_pct            | 0.0533 | DEGRADED |            0.0428 |
| em_pct            | 0.0596 | DEGRADED |            0.049  |
| platanus_pct      | 0.0205 | OK       |            0.0099 |
| cell_vpa_score    | 0.0106 | OK       |            0      |
| species_richness  | 0.0106 | OK       |            0.0001 |
| total_trees       | 0.0106 | OK       |            0      |
| trees_young_pct   | 0.0106 | OK       |            0      |


## Cell A4 — the kill: an external test against data the model never saw

The Session-4 number says the model reproduces *its own composite*. It says nothing about
whether the **ecological claim** is true. So we built an independent target — observed GBIF
fungal occurrences (never used anywhere in the pipeline) — and asked, on the 99 cells with
≥1 record: *after the abiotic null (sealed + greenness + sampling effort), does the
biotic/host block add any explanatory power for real fungal richness?* Pre-registered pass:
ΔAdj-R² ≥ 0.05 **and** partial-F p < 0.05.

In [6]:
# Cell A4 — external GBIF validation (reproduces outputs/phase-5/external_validation_results.json)
g = grid if grid.crs is not None else grid.set_crs(ev.GRID_CRS)
tgt = ev.build_target(g)
df = g.merge(tgt, on="cell_id", how="left").copy()
df["log_effort"] = np.log1p(df["gbif_effort"])
for c in set(ev.ABIOTIC) | set(ev.BIOTIC):
    if c in df and df[c].isna().any(): df[c] = df[c].fillna(df[c].median())
obs = df[df["gbif_effort"] >= 1].reset_index(drop=True)
y = obs["gbif_richness"].to_numpy(float)
M0 = ev.ols(obs, y, ev.ABIOTIC)               # abiotic null
M1 = ev.ols(obs, y, ev.ABIOTIC + ev.BIOTIC)   # + biotic / host block
F, p = ev.partial_f(M0, M1)
vifs = ev.vif(obs, ev.ABIOTIC + ev.BIOTIC)
print(f"observed cells: {len(obs)} / {len(df)}")
print(f"M0 abiotic null  Adj-R2 = {M0['adj_r2']:.4f}")
print(f"M1 + biotic/host Adj-R2 = {M1['adj_r2']:.4f}")
print(f"Delta Adj-R2 = {M1['adj_r2']-M0['adj_r2']:+.4f}   (pre-registered pass needs >= +0.05)")
print(f"partial-F = {F:.3f},  p = {p:.5f}   (pass needs p < 0.05)")
print(f"\nVERDICT: {'PASS' if (M1['adj_r2']-M0['adj_r2']>=0.05 and p<0.05) else 'FAIL'} "
      f"-> the biotic/host block adds NO signal beyond the abiotic surface.")
print(f"Collinearity (VIF): platanus_pct={vifs['platanus_pct']:.0f}, prpi={vifs['prpi']:.0f}, "
      f"am_pct/em_pct=inf -> the 'five components' are not five independent signals.")

observed cells: 99 / 494
M0 abiotic null  Adj-R2 = 0.6972
M1 + biotic/host Adj-R2 = 0.6777
Delta Adj-R2 = -0.0195   (pre-registered pass needs >= +0.05)
partial-F = 0.178,  p = 0.98917   (pass needs p < 0.05)

VERDICT: FAIL -> the biotic/host block adds NO signal beyond the abiotic surface.
Collinearity (VIF): platanus_pct=699, prpi=628, am_pct/em_pct=inf -> the 'five components' are not five independent signals.


In [7]:
# Cell A5 — confidence on the model: it is NOT a single number
spread = (preds.groupby("split")[["y_true","y_pred__LinearRegression"]]
    .apply(lambda d: pd.Series(metrics(d["y_true"], d["y_pred__LinearRegression"]))).round(4))
print("R2/MAE by split (the generalization story):")
print(spread.to_markdown())
print(f"\nMAE degrades {0.0106/0.0017:.0f}x from eval (0.0017) to held-out test (0.0106): the "
      "honest spatial-generalization cost.")
print(f"Stability (from outputs/phase-4/stability.json): "
      f"noise-injection test-R2 {stability['noise_sigma0.02_test_r2']} "
      f"(delta {stability['noise_delta']}); alt-seed test-R2 {stability['alt_seed_test_r2']}.")
print("Reading: the model is internally stable, but it recovers the composite, not ecology — "
      "and the composite is ~a re-skin of sealed surface (convergent r(pred,sealed)=0.94).")

R2/MAE by split (the generalization story):
| split   |     R2 |    MAE |   RMSE |
|:--------|-------:|-------:|-------:|
| eval    | 0.9991 | 0.0017 | 0.0039 |
| test    | 0.8769 | 0.0106 | 0.0509 |
| train   | 0.9997 | 0.001  | 0.0022 |

MAE degrades 6x from eval (0.0017) to held-out test (0.0106): the honest spatial-generalization cost.
Stability (from outputs/phase-4/stability.json): noise-injection test-R2 0.8761 (delta -0.0008); alt-seed test-R2 {'1': 0.8774, '7': 0.8774, '123': 0.8769}.
Reading: the model is internally stable, but it recovers the composite, not ecology — and the composite is ~a re-skin of sealed surface (convergent r(pred,sealed)=0.94).


---
# CYCLE B (Track B) — the product we shipped

**The claim under evaluation (one falsifiable sentence):**
> *Ranking Barcelona's 400 m cells by (mature-plane-pollen source × residential exposure)
> re-orders the city's plane-removal sequence relative to the naive "remove where planes are
> densest" rule, captures more modeled allergen-exposure relief per removal, and that
> re-ordering survives reasonable perturbation.*

- **Decision it serves:** Espais Verds sequences the already-decided plane reduction
  (Pla Director 2017–2037, 27%→<12%) so each removal relieves the most exposure.
- **The exact evidence:** the four pre-registered tests T1–T4 below, recomputed from
  `data/processed/allergen_layers.parquet`.

In [8]:
# Cell B1 — reproduce priority; T1: does exposure re-order vs naive density?
src  = layers["source_std"].to_numpy(float)
expo = layers["exposure_std"].to_numpy(float)
priority = src * expo
total = priority.sum()
sp  = float(spearmanr(priority, src).statistic)
j15 = ap.jaccard(ap.topk(priority,15), ap.topk(src,15))
j50 = ap.jaccard(ap.topk(priority,50), ap.topk(src,50))
print(f"T1  Spearman(priority, source) = {sp:.4f}")
print(f"    top-15 Jaccard = {j15:.4f}   top-50 Jaccard = {j50:.4f}")
print(f"    exposure materially re-orders (J15<0.70 AND rho<0.90): {bool(j15<0.70 and sp<0.90)}")

# worked example — accounting for people flips the naive order
ex = layers.assign(priority=priority).sort_values("priority", ascending=False)
cols = ["district","n_platanus","exposure_pop","priority"]
print("\nTop cells (people change the order vs density-only):")
print(ex[cols].head(3).round(3).to_markdown(index=False))

T1  Spearman(priority, source) = 0.8909
    top-15 Jaccard = 0.3043   top-50 Jaccard = 0.3889
    exposure materially re-orders (J15<0.70 AND rho<0.90): True

Top cells (people change the order vs density-only):
| district   |   n_platanus |   exposure_pop |   priority |
|:-----------|-------------:|---------------:|-----------:|
| NOU BARRIS |          251 |       13435.7  |      0.49  |
| SANT MARTÍ |          485 |        6501.35 |      0.463 |
| EIXAMPLE   |          197 |       12692.9  |      0.367 |


In [9]:
# Cell B2 — T2: redundancy — two layers, or one variable in a costume?
cps = float(np.corrcoef(priority, src)[0,1])
cpe = float(np.corrcoef(priority, expo)[0,1])
cse = float(np.corrcoef(src,  expo)[0,1])
print(f"T2  corr(priority, source)   = {cps:.4f}")
print(f"    corr(priority, exposure) = {cpe:.4f}")
print(f"    corr(source,  exposure)  = {cse:.4f}")
print(f"    both layers material (>=0.3 each): {bool(abs(cps)>=0.3 and abs(cpe)>=0.3)}; "
      f"inputs not collinear (<0.8): {bool(abs(cse)<0.8)}")
print("    -> NOT one variable in a costume: source and exposure are nearly independent (0.30),")
print("       and both move the ranking. (Contrast Cycle A, where the composite ~= sealed alone.)")

T2  corr(priority, source)   = 0.8030
    corr(priority, exposure) = 0.6361
    corr(source,  exposure)  = 0.2975
    both layers material (>=0.3 each): True; inputs not collinear (<0.8): True
    -> NOT one variable in a costume: source and exposure are nearly independent (0.30),
       and both move the ranking. (Contrast Cycle A, where the composite ~= sealed alone.)


In [10]:
# Cell B3 — T3 burden captured vs baselines  +  T4 sensitivity (robustness cuts)
def bc(order, k): return float(priority[np.argsort(-order)[:k]].sum()/total)
def rand_bc(k, draws=200):
    rng = np.random.default_rng(RANDOM_SEED); n=len(priority)
    return float(np.mean([priority[rng.choice(n,k,replace=False)].sum()/total for _ in range(draws)]))
print("T3  modeled exposure-relief burden captured by the top-k:")
for k in (15,50):
    pri, den, rnd = bc(priority,k), bc(src,k), rand_bc(k)
    print(f"    k={k:2d}:  priority {pri:.4f}  vs density-only {den:.4f}  vs random {rnd:.4f}"
          f"   -> margin over the city's rule {pri-den:+.4f}")

def reorders(pri, order):
    return bool(ap.jaccard(ap.topk(pri,15), ap.topk(order,15)) < 0.70
                and spearmanr(pri, order).statistic < 0.90)
dens = ap.minmax(layers["plane_density"].to_numpy(float))
src_rank = ap.minmax(pd.Series(src).rank().to_numpy())
expo_rank = ap.minmax(pd.Series(expo).rank().to_numpy())
t4 = {"uniform_maturity": reorders(dens*expo, dens),
      "rank_normalized": reorders(src_rank*expo_rank, src_rank),
      "min_aggregation": reorders(np.minimum(src,expo), src)}
print(f"\nT4  re-order verdict under perturbation: {t4}  -> holds {sum(t4.values())}/3")

T3  modeled exposure-relief burden captured by the top-k:
    k=15:  priority 0.1801  vs density-only 0.1337  vs random 0.0296   -> margin over the city's rule +0.0464
    k=50:  priority 0.4458  vs density-only 0.3524  vs random 0.1015   -> margin over the city's rule +0.0935

T4  re-order verdict under perturbation: {'uniform_maturity': True, 'rank_normalized': True, 'min_aggregation': True}  -> holds 3/3


In [11]:
# Cell B4 — equity variant + the layers we REJECTED (anti-cherry-picking)
dep = layers["deprivation_std"].to_numpy(float)
print(f"EQUITY v3 precondition — deprivation decorrelated from both layers:")
print(f"    corr(deprivation, source) = {np.corrcoef(dep,src)[0,1]:+.4f}; "
      f"corr(deprivation, exposure) = {np.corrcoef(dep,expo)[0,1]:+.4f}  -> genuine new info")
burden = src*expo
def bcap(order,k): return float(burden[np.argsort(-order)[:k]].sum()/burden.sum())
terc = pd.qcut(layers["cell_income"],3,labels=["low","mid","high"])
deprived = set(layers.loc[terc=="low","cell_id"]); cid = layers["cell_id"].to_numpy()
def share(order,k):
    top = cid[np.argsort(-order)[:k]]; return float(np.mean([c in deprived for c in top]))
e15, q15 = bcap(burden,15), bcap(burden*dep,15)
print(f"    top-15: efficiency map captures {e15:.4f}, equity map {q15:.4f} "
      f"(sacrifice {e15-q15:.4f}); deprived-tercile share {share(burden,15):.2f} -> {share(burden*dep,15):.2f}")
print(f"    -> equity lifts the poorest-third share 40%->60% for ~0.5pp of relief. Planner chooses.\n")

# Cuts that did NOT support adding a layer (reported, not hidden):
atr = layers["at_risk_std"].to_numpy(float)
j_v2 = ap.jaccard(ap.topk(src*atr,15), ap.topk(burden,15))
sp_atr = float(spearmanr(layers["at_risk_pop"], layers["exposure_pop"]).statistic)
print(f"REJECTED age-prevalence layer: Spearman(at_risk_pop, population) = {sp_atr:.3f}, "
      f"top-15 Jaccard vs v1 = {j_v2:.3f} -> redundant with population, does NOT re-order. Dropped.")
print("REJECTED sex weighting: women 1.62x antihistamine use, but sex ratio ~constant across "
      "neighbourhoods -> no mappable layer. REJECTED bike-exposure: no cyclist-volume data, no "
      "validation path. (Reporting these is the point: a layer that can't re-order is a finding.)")

EQUITY v3 precondition — deprivation decorrelated from both layers:
    corr(deprivation, source) = -0.0077; corr(deprivation, exposure) = +0.1733  -> genuine new info
    top-15: efficiency map captures 0.1801, equity map 0.1748 (sacrifice 0.0052); deprived-tercile share 0.40 -> 0.60
    -> equity lifts the poorest-third share 40%->60% for ~0.5pp of relief. Planner chooses.

REJECTED age-prevalence layer: Spearman(at_risk_pop, population) = 0.999, top-15 Jaccard vs v1 = 0.875 -> redundant with population, does NOT re-order. Dropped.
REJECTED sex weighting: women 1.62x antihistamine use, but sex ratio ~constant across neighbourhoods -> no mappable layer. REJECTED bike-exposure: no cyclist-volume data, no validation path. (Reporting these is the point: a layer that can't re-order is a finding.)


---
# The verdict

**Cycle A — STOP (the mycorrhizal model is falsified), ~high confidence.** Three independent
lines converge: (a) the headline composite is ~91% sealed surface, ecological components at
effective weight ≈ 0; (b) the external GBIF test returns a flat null — the biotic/host block
adds ΔAdj-R² = −0.0195, partial-F **p = 0.989**; (c) a 44-source literature review finds the
AM→EM host lever weak-to-unsupported. We did **not** relabel a sealed-surface map as a fungal
one. We stopped the ecological claim.

**Cycle B — SHIP ~75% (analytically), DEPLOY-PENDING.** The allergen product passes all six
pre-registered build criteria (T1–T4 + equity precondition), survives every sensitivity
perturbation, and is honest about its one un-closable limitation (no measured pollen). The
~75% and the *deploy-pending* both come from the same place: deployment is gated on a real
stakeholder Monday-test and an independent reproduction — **Phase-6 work, deferred to class**
(`phase-6/phase-5-audit.md §3, §8`).

**Compared to what.** Cycle A model vs three baselines: test R² 0.877 vs best baseline −0.29 —
it wins the Phase-4 contest yet still fails Phase 5, which is the whole lesson. Cycle B vs the
city's implicit density-only rule: ~70% of the top-15 change, +0.046 burden captured at top-15.

**What we are NOT claiming (≥3):**
1. **NOT** that the map is validated against measured pollen — it is a literature-anchored
   emission proxy (the central limitation).
2. **NOT** a health/allergy *outcome* predictor — it ranks *exposure potential*.
3. **NOT** a decision on *whether* to remove planes — policy decides that; we only sequence it.
4. **NOT** valid below 400 m nor for within-cell siting (MAUP on grid + areal population).
5. **NOT** that the AM→EM mycorrhizal mechanism holds — it is here *falsified* in this data.

## Process review — what we would redo if we started Monday

- **Weakest link:** Cycle A validated the composite against *its own ingredients* (eval R²
  0.999). That is why a "successful" model was meaningless until the **external** test. The
  single most valuable move was building a target the pipeline never saw.
- **The shortcut (knowing):** maturity = cell-level young-tree share, not per-tree trunk
  diameter; residential population as a daytime-exposure proxy (no mobility data).
- **Untested assumption carried from earlier sessions:** that AM→EM host replacement reduces a
  meaningful "mismatch" — inherited from Phase 1, never validated, finally falsified in Cycle A.
- **The thing we avoided looking at, then looked at:** the effective weights inside
  `composite_score_B`. Looking showed sealed surface dominates and the ecological signal was
  weighted into irrelevance — the finding that forced the pivot.

**Loopback taken:** Phase-5 evaluation → Phase-1 re-frame (kill the ecological claim, keep the
abiotic+inventory signal the data supports) → a full second CRISP-DM cycle for the allergen
product. A documented, pre-registered hypothesis death is the clearest demonstration of the
CRISP-DM iteration loop the course teaches.

## Reproducibility check

This notebook restart-runs top-to-bottom with no errors on the `hermes-agent` kernel, and every
number it prints is the source for the matching number in `docs/evaluation-report.md`. The
underlying analyses are the canonical scripts: `src/external_validation.py` (the kill),
`src/allergen_priority.py` (T1–T4), `src/equity_layer.py` (v3), `src/phase5_robustness.py`
(model stability). Determinism: `RANDOM_SEED = 42` throughout; the test cluster was frozen at
split time and inspected once.